# 🌍 Earthquake LSTM Training - Google Colab

## 📋 Overview
Train PyTorch LSTM model for earthquake prediction with GPU acceleration.

## 🚀 Quick Start
1. **Upload data**: `dongdat.csv` (or `features_lstm.csv`)
2. **Run all cells** below
3. **Download model** from Google Drive

## 1️⃣ Setup & Check GPU

In [ ]:
# Check GPU
!nvidia-smi

# Install dependencies
!pip install -q pandas numpy scikit-learn tqdm

import os
import sys
import warnings
warnings.filterwarnings('ignore')

import torch
print(f"\n{'='*70}")
print(' GPU INFO')
print(f"{'='*70}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"{'='*70}\n")

## 2️⃣ Configuration & Setup

In [ ]:
# Training Configuration - EDIT HERE!
CONFIG = {
    # Data
    'use_drive': True,           # Use Google Drive for data
    'data_path': '/content/drive/MyDrive/features_lstm.csv',  # Path in Drive
    
    # Training
    'epochs': 50,
    'batch_size': 128,           # T4: 64, L4: 128, A100: 256
    'learning_rate': 0.001,
    'hidden_units': [128, 64],   # LSTM hidden layers
    'dropout': 0.3,
    'patience': 15,              # Early stopping
    
    # Data filtering
    'min_events': 500,           # Min events per region
    'single_region': None,       # Or specify like 'R257_114'
}

# Paths
WORK_DIR = '/content/earthquake_training'
MODEL_DIR = f'{WORK_DIR}/models'

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"✓ Working directory: {WORK_DIR}")
print(f"✓ Model directory: {MODEL_DIR}")

## 3️⃣ Mount Google Drive & Load Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Mount Google Drive
if CONFIG['use_drive']:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Check if data exists
    data_path = CONFIG['data_path']
    if not os.path.exists(data_path):
        print(f"⚠️  Data not found at: {data_path}")
        print("Please upload your data file to Google Drive first!")
        print("\nOptions:")
        print("1. Upload features_lstm.csv to Google Drive")
        print("2. Or change 'data_path' in CONFIG above")
        print("3. Or set 'use_drive': False and upload directly to Colab")
    else:
        print(f"✓ Data found: {data_path}")
        print(f"  Size: {os.path.getsize(data_path) / 1024**3:.2f} GB")
else:
    print("⚠️  Please upload data file manually")
    data_path = None

## 4️⃣ Define All Training Code

In [ ]:
import pickle
import json
from datetime import datetime
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Constants
SEQUENCE_LENGTH = 5

# Features
ORIGINAL_FEATURES = ['time', 'latitude', 'longitude', 'depth', 'mag',
                     'sig', 'mmi', 'cdi', 'felt', 'region_code']
CORE_FEATURES = ['is_aftershock', 'mainshock_mag', 'seismicity_density_100km',
                 'coulomb_stress_proxy', 'regional_b_value']
SEQUENCE_FEATURES = ['sequence_id', 'seq_position', 'is_seq_mainshock',
                     'seq_mainshock_mag', 'seq_length', 'time_since_seq_start_sec']
LSTM_FEATURES = ['time_since_last_event', 'time_since_last_M5',
                 'interval_lag1', 'interval_lag2', 'interval_lag3',
                 'interval_lag4', 'interval_lag5']
TARGET_FEATURES = ['target_time_to_next', 'target_next_mag', 'target_next_mag_binary']

INPUT_FEATURES = [f for f in (ORIGINAL_FEATURES + CORE_FEATURES + SEQUENCE_FEATURES + LSTM_FEATURES) if f != 'time']

print("✓ All imports successful")
print(f"  Input features: {len(INPUT_FEATURES)}")

In [ ]:
# Dataset Class
class EarthquakeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y.astype(np.float32))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# LSTM Model
class EarthquakeLSTM(nn.Module):
    def __init__(self, n_features, lstm_hidden=[128, 64], dropout=0.3):
        super(EarthquakeLSTM, self).__init__()
        self.n_features = n_features
        self.lstm_hidden = lstm_hidden

        # LSTM layers
        self.lstm_layers = nn.ModuleList()
        lstm_input_size = n_features
        for i, hidden_size in enumerate(lstm_hidden):
            self.lstm_layers.append(
                nn.LSTM(input_size=lstm_input_size, hidden_size=hidden_size,
                       num_layers=1, batch_first=True,
                       dropout=dropout if i < len(lstm_hidden) - 1 else 0)
            )
            lstm_input_size = hidden_size

        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_hidden[-1], 64)
        self.relu = nn.ReLU()
        self.bn1 = nn.BatchNorm1d(64)

        # Output heads
        self.output_time = nn.Linear(64, 1)
        self.output_mag = nn.Linear(64, 1)
        self.output_binary = nn.Linear(64, 1)

    def forward(self, x):
        batch_size = x.size(0)

        # LSTM layers
        lstm_out = x
        for lstm in self.lstm_layers:
            lstm_out, _ = lstm(lstm_out)

        lstm_out = lstm_out[:, -1, :]
        lstm_out = self.dropout(lstm_out)

        out = self.fc1(lstm_out)
        out = self.relu(out)
        out = self.bn1(out)
        out = self.dropout(out)

        time_to_next = torch.relu(self.output_time(out))
        next_mag = self.output_mag(out).squeeze(-1)
        next_mag_binary = torch.sigmoid(self.output_binary(out)).squeeze(-1)

        return time_to_next.squeeze(-1), next_mag, next_mag_binary

print("✓ Model classes defined")

In [ ]:
# Data Preparation Functions
def prepare_data(data_path, min_events=500, single_region=None):
    """Load and prepare data for training"""
    print(f"\n{'='*70}")
    print(' LOADING DATA')
    print(f"{'='*70}")
    
    # Load data
    df = pd.read_csv(data_path)
    df['time'] = pd.to_datetime(df['time'])
    print(f"  Total events: {len(df):,}")
    print(f"  Features: {len(df.columns)}")
    
    # Filter by region
    if single_region:
        print(f"\nMode: SINGLE REGION - {single_region}")
        df = df[df['region_code'] == single_region].copy()
        if len(df) < min_events:
            raise ValueError(f"Region has only {len(df)} events, need {min_events}")
    else:
        print(f"\nMode: ALL REGIONS (min {min_events} events/region)")
        region_counts = df['region_code'].value_counts()
        valid_regions = region_counts[region_counts >= min_events].index
        df = df[df['region_code'].isin(valid_regions)].copy()
        print(f"  Valid regions: {len(valid_regions)}")
        print(f"  Total events: {len(df):,}")
    
    return df

def create_sequences(df, sequence_length=5):
    """Create sequences for LSTM"""
    print(f"\n  Creating sequences...")
    
    feature_cols = [col for col in INPUT_FEATURES if col != 'time']
    
    # Encode region
    le = LabelEncoder()
    df['region_encoded'] = le.fit_transform(df['region_code'])
    
    sequences = []
    targets = []
    
    for i in tqdm(range(sequence_length, len(df)), desc="  Sequences"):
        seq_data = df.iloc[i-sequence_length:i]
        
        seq_features = []
        for feat in feature_cols:
            if feat == 'region_code':
                seq_features.append(seq_data['region_encoded'].values)
            elif feat in seq_data.columns:
                seq_features.append(seq_data[feat].values)
        
        seq_array = np.column_stack(seq_features)
        sequences.append(seq_array)
        
        target = df.iloc[i][TARGET_FEATURES].values
        targets.append(target)
    
    X = np.array(sequences)
    y = np.array(targets)
    
    print(f"  ✓ X shape: {X.shape}")
    print(f"  ✓ y shape: {y.shape}")
    
    return X, y

def split_and_scale(X, y, train_ratio=0.8, val_ratio=0.1):
    """Split and scale data"""
    print(f"\n  Splitting data...")
    
    n = len(X)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    
    X_train, y_train = X[:n_train], y[:n_train]
    X_val, y_val = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
    X_test, y_test = X[n_train+n_val:], y[n_train+n_val:]
    
    print(f"  Train: {len(X_train):,}")
    print(f"  Val: {len(X_val):,}")
    print(f"  Test: {len(X_test):,}")
    
    # Scale
    print(f"\n  Scaling features...")
    scaler = StandardScaler()
    
    X_train_2d = X_train.reshape(-1, X_train.shape[-1])
    X_train_scaled = scaler.fit_transform(X_train_2d).reshape(X_train.shape)
    
    X_val_2d = X_val.reshape(-1, X_val.shape[-1])
    X_val_scaled = scaler.transform(X_val_2d).reshape(X_val.shape)
    
    X_test_2d = X_test.reshape(-1, X_test.shape[-1])
    X_test_scaled = scaler.transform(X_test_2d).reshape(X_test.shape)
    
    print(f"  ✓ Scaling complete")
    
    return (X_train_scaled, X_val_scaled, X_test_scaled,
            y_train, y_val, y_test, scaler)

print("✓ Data functions defined")

In [ ]:
# Training Class
class EarthquakeTrainer:
    def __init__(self, model, device, learning_rate=0.001):
        self.model = model.to(device)
        self.device = device
        self.optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=5
        )
        self.criterion_mse = nn.MSELoss()
        self.criterion_bce = nn.BCELoss()
        
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_time_mae': [], 'val_time_mae': [],
            'train_mag_mae': [], 'val_mag_mae': [],
            'train_binary_acc': [], 'val_binary_acc': []
        }
        self.best_val_loss = float('inf')
        self.best_model_state = None

    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = total_time_mae = total_mag_mae = 0
        total_binary_correct = total_samples = 0
        
        pbar = tqdm(train_loader, desc="  Training", leave=False)
        for X_batch, y_batch in pbar:
            X_batch = X_batch.to(self.device)
            y_batch = y_batch.to(self.device)
            
            time_pred, mag_pred, binary_pred = self.model(X_batch)
            
            loss = (self.criterion_mse(time_pred, y_batch[:, 0]) +
                   self.criterion_mse(mag_pred, y_batch[:, 1]) +
                   self.criterion_bce(binary_pred, y_batch[:, 2]))
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item() * X_batch.size(0)
            total_time_mae += torch.abs(time_pred - y_batch[:, 0]).sum().item()
            total_mag_mae += torch.abs(mag_pred - y_batch[:, 1]).sum().item()
            total_binary_correct += ((binary_pred > 0.5) == (y_batch[:, 2] > 0.5)).sum().item()
            total_samples += X_batch.size(0)
        
        return {
            'loss': total_loss / total_samples,
            'time_mae': total_time_mae / total_samples,
            'mag_mae': total_mag_mae / total_samples,
            'binary_acc': total_binary_correct / total_samples
        }

    def validate(self, val_loader):
        self.model.eval()
        total_loss = total_time_mae = total_mag_mae = 0
        total_binary_correct = total_samples = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)
                
                time_pred, mag_pred, binary_pred = self.model(X_batch)
                
                loss = (self.criterion_mse(time_pred, y_batch[:, 0]) +
                       self.criterion_mse(mag_pred, y_batch[:, 1]) +
                       self.criterion_bce(binary_pred, y_batch[:, 2]))
                
                total_loss += loss.item() * X_batch.size(0)
                total_time_mae += torch.abs(time_pred - y_batch[:, 0]).sum().item()
                total_mag_mae += torch.abs(mag_pred - y_batch[:, 1]).sum().item()
                total_binary_correct += ((binary_pred > 0.5) == (y_batch[:, 2] > 0.5)).sum().item()
                total_samples += X_batch.size(0)
        
        return {
            'loss': total_loss / total_samples,
            'time_mae': total_time_mae / total_samples,
            'mag_mae': total_mag_mae / total_samples,
            'binary_acc': total_binary_correct / total_samples
        }

    def train(self, train_loader, val_loader, epochs, patience=15):
        print(f"\n{'='*70}")
        print(' TRAINING')
        print(f"{'='*70}")
        print(f"Device: {self.device}")
        print(f"Epochs: {epochs}")
        print(f"Train samples: {len(train_loader.dataset):,}")
        print(f"Val samples: {len(val_loader.dataset):,}")
        print(f"{'='*70}\n")
        
        patience_counter = 0
        pbar = tqdm(range(epochs), desc="Epochs")
        
        for epoch in pbar:
            train_metrics = self.train_epoch(train_loader)
            val_metrics = self.validate(val_loader)
            
            # Update history
            for k in self.history:
                self.history[f'train_{k.split("_")[1]}'].append(train_metrics[k.split("_")[1]])
                self.history[f'val_{k.split("_")[1]}'].append(val_metrics[k.split("_")[1]])
            
            pbar.set_description(
                f"Epoch {epoch+1} - Val Loss: {val_metrics['loss']:.4f} - "
                f"Time MAE: {val_metrics['time_mae']:.0f}s - "
                f"Mag MAE: {val_metrics['mag_mae']:.2f}"
            )
            
            self.scheduler.step(val_metrics['loss'])
            
            if val_metrics['loss'] < self.best_val_loss:
                self.best_val_loss = val_metrics['loss']
                self.best_model_state = self.model.state_dict().copy()
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
        
        if self.best_model_state:
            self.model.load_state_dict(self.best_model_state)
            print(f"\n✓ Best model loaded (val_loss: {self.best_val_loss:.4f})")
        
        return self.history

print("✓ Trainer class defined")

## 5️⃣ Prepare Data

In [ ]:
# Load and prepare data
df = prepare_data(
    data_path,
    min_events=CONFIG['min_events'],
    single_region=CONFIG['single_region']
)

# Create sequences
X, y = create_sequences(df, sequence_length=SEQUENCE_LENGTH)

# Split and scale
X_train, X_val, X_test, y_train, y_val, y_test, scaler = split_and_scale(X, y)

## 6️⃣ Create DataLoaders

In [ ]:
# Create datasets
train_dataset = EarthquakeDataset(X_train, y_train)
val_dataset = EarthquakeDataset(X_val, y_val)
test_dataset = EarthquakeDataset(X_test, y_test)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2
)

print(f"\n✓ DataLoaders created")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 7️⃣ Create Model & Start Training

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create model
n_features = X_train.shape[-1]
model = EarthquakeLSTM(
    n_features=n_features,
    lstm_hidden=CONFIG['hidden_units'],
    dropout=CONFIG['dropout']
)

print(f"\n{'='*70}")
print(' MODEL ARCHITECTURE')
print(f"{'='*70}")
print(f"Input features: {n_features}")
print(f"Sequence length: {SEQUENCE_LENGTH}")
print(f"LSTM hidden: {CONFIG['hidden_units']}")
print(f"Dropout: {CONFIG['dropout']}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")
print(f"{'='*70}\n")

# Create trainer
trainer = EarthquakeTrainer(
    model=model,
    device=device,
    learning_rate=CONFIG['learning_rate']
)

# Train
history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=CONFIG['epochs'],
    patience=CONFIG['patience']
)

## 8️⃣ Evaluate & Save Model

In [ ]:
# Test evaluation
print(f"\n{'='*70}")
print(' TEST EVALUATION')
print(f"{'='*70}")

test_metrics = trainer.validate(test_loader)
print(f"Test Loss: {test_metrics['loss']:.4f}")
print(f"Test Time MAE: {test_metrics['time_mae']:.1f} seconds")
print(f"Test Mag MAE: {test_metrics['mag_mae']:.3f}")
print(f"Test Binary Acc: {test_metrics['binary_acc']:.3f}")

# Save model
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
region_suffix = f"_{CONFIG['single_region']}" if CONFIG['single_region'] else "_all"

model_path = f"{MODEL_DIR}/model{region_suffix}_{timestamp}.pt"
scaler_path = f"{MODEL_DIR}/scaler{region_suffix}_{timestamp}.pkl"
history_path = f"{MODEL_DIR}/history{region_suffix}_{timestamp}.json"

# Save checkpoint
checkpoint = {
    'model_state_dict': model.state_dict(),
    'n_features': n_features,
    'lstm_hidden': CONFIG['hidden_units'],
    'best_val_loss': trainer.best_val_loss,
    'test_metrics': test_metrics,
    'config': CONFIG
}
torch.save(checkpoint, model_path)

# Save scaler
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

# Save history
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print(f"\n{'='*70}")
print(' SAVED FILES')
print(f"{'='*70}")
print(f"Model: {model_path}")
print(f"Scaler: {scaler_path}")
print(f"History: {history_path}")
print(f"{'='*70}")

## 9️⃣ Copy to Google Drive

In [ ]:
if CONFIG['use_drive']:
    import shutil
    
    drive_backup_dir = f"/content/drive/MyDrive/earthquake_models/{timestamp}"
    os.makedirs(drive_backup_dir, exist_ok=True)
    
    # Copy all files
    for file_path in [model_path, scaler_path, history_path]:
        shutil.copy(file_path, drive_backup_dir)
    
    print(f"\n✓ All files copied to Google Drive:")
    print(f"  {drive_backup_dir}")
else:
    print("\n⚠️  Google Drive not enabled. Download files manually from Colab.")

## 🎉 Training Complete!

### Next Steps:
1. **Download model** from Google Drive or Colab files
2. **Use for prediction** in your local environment
3. **Experiment** with different hyperparameters

### Model Files:
- `model_*.pt` - Trained model checkpoint
- `scaler_*.pkl` - Feature scaler
- `history_*.json` - Training history

### To Use Model for Prediction:
```python
import torch
import pickle

# Load model
checkpoint = torch.load('model_*.pt')
model = EarthquakeLSTM(**checkpoint)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Load scaler
with open('scaler_*.pkl', 'rb') as f:
    scaler = pickle.load(f)
```